In [3]:
import numpy as np
import matplotlib.pyplot as plt
# import mkl

np.random.seed(1234)
# mkl.set_num_threads(2)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams["figure.figsize"] = [16, 9]

## Utility functions

In [ ]:
def append_ones(matrix, axis=1):
    ones = np.ones((matrix.shape[0], 1), dtype=matrix.dtype)
    return np.concatenate((matrix, ones), axis=axis)

In [16]:
def append_ones_v2(matrix, axis=1):
    if axis == 1:
        ones = np.ones((matrix.shape[0], 1), dtype=matrix.dtype)
    else:
        ones = np.ones((1, matrix.shape[0]), dtype=matrix.dtype)

    return np.concatenate((matrix, ones), axis=axis)

In [17]:
def test_append_ones():
    matrix_a = np.array([
        [1,2,3],
        [4,5,6],
        [7,8,9]
    ])
    matrix_a_result_1 = append_ones_v2(matrix_a, 1)
    print(matrix_a_result_1)

    matrix_a_result_2 = append_ones_v2(matrix_a, 0)
    print(matrix_a_result_2)

test_append_ones()

[[1 2 3 1]
 [4 5 6 1]
 [7 8 9 1]]
[[1 2 3]
 [4 5 6]
 [7 8 9]
 [1 1 1]]


## Feed-forward, activations and tiles

In [18]:
# one layer feed forward
def feed_forward(W, dataset, activation_fun):
    return activation_fun(dataset @ W)

In [19]:
# sigmoid activation function
def sigmoid(matrix):
    return 1 / (1 + np.exp(-matrix))

In [26]:
# arrange 2D matrices as tiles (takes 4D `examples` tensor with dims: rows x cols x tile_height x tile_width)
def tiles(examples, space=2):
    rows, cols, tile_height, tile_width = examples.shape
    matrix = np.zeros((
        rows * (tile_height + space) - space,
        cols * (tile_width + space) - space
    ))

    for y in range(rows):
        for x in range(cols):
            start_y = y * (tile_height + space)
            start_x = x * (tile_width + space)
            matrix[start_y:start_y + tile_height, start_x:start_x + tile_width] = examples[y, x]

    return matrix

## Function tests

In [21]:
def test_sigmoid():
    matrix_a = np.array([1,2,3])
    matrix_b = np.array([
        [1,2,3],
        [4,5,6]
    ])

    print(sigmoid(matrix_a))
    print(sigmoid(matrix_b))

test_sigmoid()

[0.73105858 0.88079708 0.95257413]
[[0.73105858 0.88079708 0.95257413]
 [0.98201379 0.99330715 0.99752738]]


In [29]:
def test_tiles():
    examples = np.ones((4,4,5,5))
    image = tiles(examples, space = 1)
    plt.imshow(image, cmap='gray')
    plt.show()

# test_tiles()

## Histogram of activations and filters plot

In [30]:
class Rbm:
    def __init__(self, visible_size, hidden_size, learning_rate):
        self.visible_size = visible_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.W = np.random.normal(scale=0.01, size=(visible_size+1, hidden_size+1)).astype(np.float32)
        self.W[:, -1] = 0.0
        self.W[-1, :] = 0.0

In [36]:
rbm = Rbm(6, 1, 0.1)
print(rbm.W)

[[-0.01472567  0.        ]
 [-0.00636605  0.        ]
 [-0.00387815  0.        ]
 [ 0.01134157  0.        ]
 [-0.0070552   0.        ]
 [-0.0063821   0.        ]
 [ 0.          0.        ]]


In [31]:
import mnist
import pickle
import seaborn as sns

with open("./lab1_rbm.pickle.dat", "rb") as f:
    rbm = pickle.load(f)

DATASET_SIZE = 512
DIGIT_SIZE = 28
mnist_dataset = mnist.test_images().astype(np.float32)
np.random.shuffle(mnist_dataset)
mnist_dataset = np.reshape(mnist_dataset[:DATASET_SIZE] / 255.0, newshape=(DATASET_SIZE, DIGIT_SIZE*DIGIT_SIZE))

mnist_dataset = append_ones(mnist_dataset)

ModuleNotFoundError: No module named 'mnist'

In [32]:
# Plotting mean hidden activations
activations = feed_forward(rbm.W, mnist_dataset, sigmoid)
mean_activations = np.mean(activations, 1)
sns.distplot(mean_activations, bins=50)

# Displaying RBM filetrs
filters = np.reshape(np.transpose(rbm.W)[:-1, :-1], newshape=(16, -1, 28, 28))
filters = np.clip(filters, -1.0, 1.0)

img = tiles(filters)
plt.matshow(img, cmap='gray', interpolation='none')
plt.axis('off')
plt.show()

NameError: name 'rbm' is not defined